# NPTEL Study Notes: Introduction to Large Language Models (LLMs)

## Topic: Positional Encoding, Residual Connections, and Normalization in Transformers

Welcome to the study notes for the Transformer architecture's critical sub-components. While attention mechanisms allow models to process information, they inherently lack an understanding of token order. This module covers how we inject sequence order into Transformers, stabilize their training, and optimize their architecture.

---

## 1. The Need for Positional Information

Unlike Recurrent Neural Networks (RNNs) that process tokens sequentially, Transformers access all tokens in a sequence **in parallel**.

* **The Problem:** The self-attention mechanism is permutation-invariant. Without explicit positional data, the model treats the sentence *"The dog chased the pig"* exactly the same as *"The pig chased the dog"*.
* **The Solution:** We introduce **Positional Embeddings (PE)**. A unique vector representing the token's position is added to the token's word embedding (e.g., from Word2Vec or GloVe).

> **Key Architectural Detail:** The positional embedding vector must have the **exact same dimensionality ($d_{model}$)** as the word embedding. We **sum** them together rather than concatenating them to keep the parameter count from expanding (though concatenation is theoretically possible).

### Why Naive Positional Encoding Fails

To understand modern positional encodings, it is crucial to understand why simple approaches fail:

1. **Integer Encoding (1, 2, 3...):** Assigning $1$ to the first word, $2$ to the second, etc.
* *Flaw:* The model does not know the maximum sequence length. Values can explode for long sequences, confusing the neural network.


2. **Normalized Integer Encoding:** Dividing the integer position by the maximum sequence length (e.g., 1/30, 2/30).
* *Flaw:* The absolute positional value changes depending on sequence length. The 1st position in a 3-word sentence gets a value of $0.33$, but in a 30-word sentence, it gets $0.033$. The model cannot learn a consistent representation for "Position 1".


3. **Binary Encoding (000, 001, 010):**
* *Flaw:* Word embeddings consist of continuous fractional numbers (e.g., $0.34$, $-0.12$). Adding discrete binary integers to these continuous vectors distorts the semantic meaning.



---

## 2. Sinusoidal Positional Encoding (Vaswani et al.)

To solve the above issues, the original Transformer paper introduced sinusoidal positional encoding. It uses sine and cosine functions of different frequencies to encode absolute positions.

### The Mathematical Formulation

For a specific position $pos$ and dimension $i$ within the embedding vector (where model size is $d_{model}$):

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

### Conceptual Breakdown

* **Interleaving:** Even dimensions ($0, 2, 4...$) use the Sine function. Odd dimensions ($1, 3, 5...$) use the Cosine function.
* **Frequency Variation:**
* When $i$ is small (first few elements of the vector), the frequency is **high**. High-frequency waves change rapidly, allowing the model to easily distinguish between **nearby positions** (e.g., position 1 vs. position 2).
* When $i$ is large (later elements in the vector), the frequency is **low**. Low-frequency waves change slowly, capturing macro-level positional similarities and distinguishing **distant positions** (e.g., position 1 vs. position 50).


* **Advantages:** It is independent of sequence length, guarantees unique embeddings for every position, and bounds values strictly between $[-1, 1]$.

---

## 3. Properties of Ideal Positional Embeddings

If you were to design a new positional embedding technique, it must ideally satisfy three mathematical properties:

1. **Monotonicity:** Proximity (similarity) between positional embeddings should smoothly decrease as the distance between the positions increases.
2. **Translation Invariance:** The proximity between two positions should depend only on their relative distance, not their absolute placement. The relationship between positions $x_1$ and $x_2$ should be identical to $(x_1 + m)$ and $(x_2 + m)$.
3. **Symmetry:** The proximity metric should be symmetric. Proximity($x, y$) = Proximity($y, x$).

---

## 4. Relative Positional Encoding vs. Absolute Positional Encoding

* **Absolute Encoding:** Explicitly assigns a vector based on the exact index (e.g., position 5).
* **Relative Encoding:** Argues that absolute position rarely matters in language. The syntactic relationship between *"dog"* and *"pig"* in *"The dog chased the pig"* depends entirely on their **distance from each other**, regardless of whether they appear at the beginning or end of a document. Models like Transformer-XL and T5 introduced relative position bias terms (e.g., learning a specific scalar bias $b_1$ for tokens separated by 1 step, $b_2$ for 2 steps).

---

## 5. Rotary Positional Encoding (RoPE)

RoPE (used in modern LLMs like PaLM, LLaMA, GPT-Neo) is a breakthrough that captures both absolute and relative positions simultaneously.

### The Mechanism

Instead of *adding* positional information, RoPE relies on a **multiplicative** rotation matrix. It maps the word embeddings into a complex space and rotates them.

1. Let $x_m$ be the query vector at absolute position $m$. It is transformed by multiplying it with $e^{im\theta}$, applying a rotation based on position $m$.
2. Let $x_n$ be the key vector at absolute position $n$. It is multiplied by $e^{in\theta}$.
3. When the self-attention mechanism computes the dot product between the query and the key, we multiply the query by the **complex conjugate** of the key.

**The Math (Simplified):**


$$(x_m e^{im\theta}) \cdot (x_n e^{-in\theta}) = x_m x_n e^{i(m-n)\theta}$$

### Why is this brilliant?

* We applied a transformation based entirely on **absolute positions** ($m$ and $n$).
* However, the resulting attention score organically contains $(m-n)$, which is the **relative distance**.
* **Efficiency:** The rotation matrix is sparse (mostly zeros). It operates by taking 2D chunks of the embedding vector and applying standard 2D Cartesian rotation, making it highly computationally efficient. Empirically, RoPE allows models to converge faster than standard sinusoidal embeddings.

---

## 6. Normalization and Residual Connections

Within each Transformer block (both encoder and decoder), we have specialized pathways to ensure gradients flow properly and the model remains stable.

### Residual Connections (Add)

After the self-attention sub-layer and the feed-forward sub-layer, we utilize a residual (skip) connection:


$$Output = \text{LayerNorm}(x + \text{Sublayer}(x))$$

* **Purpose:** Provides a direct shortcut for gradients to flow backward through the network, directly addressing the **vanishing gradient problem** and allowing for the training of very deep networks.

### Layer Normalization (Norm)

As vectors pass through complex attention and feed-forward operations, their statistical distributions shift (Covariance Shift). Normalization forces the values back into a stable distribution.

* **Standardization Formula:** For a given vector, we subtract its mean ($\mu$) and divide by its standard deviation ($\sigma$).

#### Batch Normalization vs. Layer Normalization

NPTEL frequently tests the distinction between these two:

| Feature | Batch Normalization | Layer Normalization (Used in Transformers) |
| --- | --- | --- |
| **How it normalizes** | Normalizes a *single feature/element* independently across all instances in a mini-batch. | Normalizes *all features/elements* across a *single instance* (vector). |
| **Batch Size Dependency** | Fails if batch size is too small (mean/variance become unrepresentative). | Completely independent of batch size. Works even for batch size = 1. |
| **Handling Sequence Lengths** | Fails with dynamic sequence lengths (if one sequence has 4 tokens and another has 8, the 5th token lacks enough batch data). | Perfect for NLP; processes each sequence token-by-token independently. |
| **Training vs. Inference** | Requires different logic during inference (uses running averages). | Uses the exact same calculations for training and inference. |

---

## 7. Transformer Training Objective Overview

Transformers are trained in an end-to-end parallel fashion using **Next Word Prediction** (for decoders/autoregressive models) or Machine Translation tasks.

* **Parallelization:** Unlike RNNs, the decoder can process the entire known ground-truth sequence during training at once. Masked self-attention ensures position $t$ can only look at positions $< t$.
* **Loss:** The output distribution (via Softmax) is compared against the actual next word using Cross-Entropy Loss. The error across all positions is summed and backpropagated simultaneously, updating all weights, including word embeddings and (if configured) trainable positional embeddings.

---

## Practice Questions for NPTEL Exam

**Q1. Why are standard integer values ($1, 2, 3...$) directly added to word embeddings not a viable method for positional encoding?**
a) They require too much memory to store.
b) The values will explode for long sequences and the maximum length is undefined.
c) They prevent the use of residual connections.
d) They force the model to look into future tokens.

> **Answer: (b)**
> **Explanation:** Integers are unbounded. In a 1000-word sequence, adding a scalar of 1000 to a word embedding whose elements are fractional (e.g., between 0 and 1) entirely destroys the semantic meaning of the word vector.

**Q2. In Sinusoidal Positional Encoding, how does the model distinguish between two tokens that are adjacent versus two tokens that are far apart?**
a) Adjacent tokens are distinguished by the low-frequency dimensions, while distant tokens rely on high-frequency dimensions.
b) Both adjacent and distant tokens are distinguished entirely by the linear bias term.
c) Adjacent tokens are distinguished by the high-frequency dimensions (early indices), while distant tokens are distinguished by low-frequency dimensions.
d) Adjacent tokens use the sine function, while distant tokens use the cosine function.

> **Answer: (c)**
> **Explanation:** High-frequency waves (small $i$) oscillate quickly, meaning the values change significantly from position 1 to position 2. Low-frequency waves change slowly, meaning they remain roughly similar for nearby tokens but differ greatly for tokens separated by large distances.

**Q3. Why is Layer Normalization preferred over Batch Normalization in Transformer models for NLP?**
a) Batch Normalization is computationally too expensive for self-attention.
b) NLP tasks involve sequences of variable lengths, which breaks Batch Normalization's feature-wise calculations across a batch.
c) Layer Normalization eliminates the need for Positional Encoding.
d) Batch Normalization causes the exploding gradient problem in recurrent models.

> **Answer: (b)**
> **Explanation:** Batch Norm calculates mean/variance for a specific feature across the batch. If sequences have different lengths, later tokens (e.g., token 20) might only exist in one or two sequences in the batch, resulting in mathematically invalid or unrepresentative normalizations. Layer Norm computes statistics across the feature dimension for a single instance, immune to sequence length variations.

**Q4. (Conceptual) What is the core mathematical advantage of Rotary Positional Encoding (RoPE) when computing the query-key dot product in self-attention?**

> **Answer & Explanation:** By applying a rotation matrix based on absolute positions ($m$ and $n$) in the complex plane, taking the dot product of the query and key inherently results in an expression that depends on $(m - n)$. This effectively injects **relative positional information** natively into the attention score without needing to manually define relative distance matrices or bias terms.

---

## Quick Revision Summary

* **Positional Embeddings (PE):** Necessary because Transformers process in parallel and lack sequence order. Added to word embeddings, requires matching dimensions ($d_{model}$).
* **Sinusoidal PE:** Uses high-frequency sine/cosine for early vector dimensions (captures local proximity) and low-frequency for later dimensions (captures global proximity).
* **Desired PE Properties:** Monotonicity, Translation Invariance, Symmetry.
* **Rotary PE (RoPE):** Multiplicative, absolute rotation encoding that naturally yields relative position dependency during the attention dot-product.
* **Residual Connections (Add):** $x + F(x)$. Prevents vanishing gradients in deep networks.
* **Layer Normalization (Norm):** Standardizes values within a single vector (mean 0, variance 1) to stabilize training and reduce covariance shift. Better than Batch Norm for variable-length NLP sequences.